# Discrete JEPA — Time Series Self-Supervised Pretraining + Forecasting

VQ-based Joint-Embedding Predictive Architecture with patch-to-patch, semantic-to-patch, and patch-to-semantic prediction tasks.

**Pipeline:** Pretrain on ETT → Forecasting fine-tune → Test

## 1 — Mount Drive & set project root

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys

# ── Change this to wherever you uploaded the allthree folder ──────────────────
PROJECT_ROOT = '/content/drive/MyDrive/allthree'

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print('Working directory:', os.getcwd())
print('Contents:', os.listdir('.'))

## 2 — Choose dataset

In [ ]:
# Options: "ettm1"  "etth1"  "etth2"  "ettm2"  "weather"  "electricity"  "traffic"
PRETRAIN_DATASET = 'ettm1'
FORECAST_DATASET = 'ettm1'   # can differ from pretrain

from dataset_registry import get_dataset_info
import pandas as pd

for label, name in [('Pretrain', PRETRAIN_DATASET), ('Forecast', FORECAST_DATASET)]:
    ds = get_dataset_info(name)
    df = pd.read_csv(ds['csv_path'], nrows=0)
    print(f'{label:8s} → {name:12s}  {ds["c_in"]} vars   {ds["csv_path"]}')

## 3 — Install dependencies

In [ ]:
import torch, sklearn
print(f'torch   {torch.__version__}')
print(f'sklearn {sklearn.__version__}')
print(f'GPU     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT available — go to Runtime > Change runtime type > T4 GPU"}')

## 4 — Verify data

In [ ]:
for label, name in [('Pretrain', PRETRAIN_DATASET), ('Forecast', FORECAST_DATASET)]:
    ds = get_dataset_info(name)
    df = pd.read_csv(ds['csv_path'])
    print(f'── {label}: {name} ──')
    print(f'  Rows    : {len(df)}')
    print(f'  Columns : {df.columns.tolist()}')
    print(df.head(2), '\n')

## 5 — (Optional) Tune config before running

Edit `Discrete_JEPA/config_files/config_pretrain.py` on Drive, or preview key values here:

In [ ]:
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'Discrete_JEPA', 'config_files'))
import importlib, config_pretrain as jepa_cfg_mod
importlib.reload(jepa_cfg_mod)

cfg = jepa_cfg_mod.config
# cfg['num_epochs'] = 100   # reduce for a quick smoke-test

print('JEPA config snapshot:')
for k in ['num_epochs', 'encoder_embed_dim', 'codebook_size', 'patch_size', 'num_patches', 'batch_size']:
    print(f'  {k:30s} = {cfg.get(k, "(not set")}')

## 6 — Run Discrete JEPA (pretrain → forecasting)

In [ ]:
from Train_and_downstream import run

run(
    model            = 'jepa',
    pretrain_dataset = PRETRAIN_DATASET,
    forecast_dataset = FORECAST_DATASET,
    skip_train       = False,
)

## 7 — Forecasting only (load existing checkpoint)

Set `skip_train=True` to skip pretraining and go straight to forecasting using a saved checkpoint.

In [ ]:
# run(model='jepa', pretrain_dataset=PRETRAIN_DATASET, forecast_dataset=FORECAST_DATASET, skip_train=True)